# 🔍 Motor de Búsqueda Semántica — Análisis
Visualización de resultados, scores de similitud y mapa de calor entre artículos.

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from IPython.display import display
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, '.')          # importar buscar.py desde el mismo directorio
from buscar import buscar, QUERIES_PRUEBA, coleccion, client, MODELO_EMBEDDING

print('✅ Todo importado correctamente')

## Parte 1 — Queries y sus resultados

In [ ]:
# Ejecutar las 4 queries y recopilar resultados
todos_resultados = {}
for q in QUERIES_PRUEBA:
    todos_resultados[q] = buscar(q, n_resultados=3)

# Mostrar resultados en formato legible
for query, resultados in todos_resultados.items():
    print(f'\n📌 "{query}"')
    print('─' * 70)
    for i, r in enumerate(resultados, 1):
        print(f'  {i}. [{r["score"]:.4f}] {r["titulo"]}')
        print(f'     {r["contenido"][:90]}...')

## Parte 2 — Tabla comparativa

In [ ]:
# Tabla: query → mejor resultado → score
filas = []
for query, resultados in todos_resultados.items():
    mejor = resultados[0]
    filas.append({
        'Query': query,
        'Mejor resultado': mejor['titulo'],
        'Score': mejor['score'],
        '2º resultado': resultados[1]['titulo'] if len(resultados) > 1 else '-',
        'Score 2º': resultados[1]['score'] if len(resultados) > 1 else '-',
    })

df_tabla = pd.DataFrame(filas)

# Estilo visual
styled = (
    df_tabla.style
    .background_gradient(subset=['Score'], cmap='YlGn')
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})
    .set_caption('📊 Resultados del motor de búsqueda semántica')
)
display(styled)

## Parte 3 — Gráfico de scores por query

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Scores de similitud semántica por query', fontsize=15, fontweight='bold', y=1.01)

colores = ['#4C9BE8', '#56C596', '#F4A261', '#E76F51']

for ax, (query, resultados), color in zip(axes.flat, todos_resultados.items(), colores):
    titulos = [r['titulo'] for r in resultados]
    scores  = [r['score']  for r in resultados]
    bars = ax.barh(titulos, scores, color=color, alpha=0.85)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Similitud coseno')
    ax.set_title(f'"{query[:45]}..."' if len(query) > 45 else f'"{query}"', fontsize=10)
    # Etiquetas de valor
    for bar, score in zip(bars, scores):
        ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('scores_por_query.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 Guardado como scores_por_query.png')

## Bonus — Mapa de calor de similitud entre artículos

In [ ]:
# Obtener todos los embeddings almacenados en ChromaDB
todos = coleccion.get(include=['embeddings', 'metadatas'])
metadatas  = todos['metadatas']
embeddings = np.array(todos['embeddings'])
titulos    = [m['titulo'] for m in metadatas]

# Calcular matriz de similitud coseno
normas = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_norm = embeddings / normas
sim_matrix = emb_norm @ emb_norm.T

# Mapa de calor
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.eye(len(titulos), dtype=bool)          # enmascarar diagonal (siempre = 1)

sns.heatmap(
    sim_matrix,
    annot=True, fmt='.2f',
    xticklabels=titulos, yticklabels=titulos,
    cmap='RdYlGn', center=0.5,
    linewidths=0.5, linecolor='white',
    mask=mask,
    ax=ax,
    annot_kws={'size': 8},
)

# Diagonal en gris
for i in range(len(titulos)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='#dddddd', lw=0))

ax.set_title('🗺️ Similitud semántica entre artículos', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('heatmap_similitud.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 Guardado como heatmap_similitud.png')

## Extra — Artículos más relacionados entre sí

In [ ]:
# Top 5 pares con mayor similitud (excluir diagonal)
np.fill_diagonal(sim_matrix, 0)
indices = np.dstack(np.unravel_index(np.argsort(sim_matrix.ravel())[::-1], sim_matrix.shape))[0]

print('🔗 Top 5 pares de artículos más similares entre sí:\n')
vistos = set()
count  = 0
for i, j in indices:
    par = tuple(sorted((i, j)))
    if par in vistos:
        continue
    vistos.add(par)
    print(f'  {count+1}. [{sim_matrix[i,j]:.4f}]')
    print(f'     • {titulos[i]}')
    print(f'     • {titulos[j]}')
    count += 1
    if count >= 5:
        break